# Two-dimensional diffusion bridge

This notebook generates `fig:generative-diffusion-2d-forward-backward`.  The data law is a three-component Gaussian mixture and the reference law is an isotropic Gaussian.  For the linear noising path
$$
    Z_t=(1-t)X+tY,
$$
the density remains a Gaussian mixture with known score.  The panels display the forward density evolution and backward probability-flow trajectories integrated from near the Gaussian endpoint back toward the data modes.

In [1]:
from pathlib import Path
import os
import sys
os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/mpl-ot4ml")
for candidate in [Path.cwd(), Path.cwd() / "notebooks-figures", Path.cwd().parent / "notebooks-figures"]:
    if (candidate / "figure_style.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        ROOT = candidate.parent if candidate.name == "notebooks-figures" else candidate
        break
else:
    raise RuntimeError("Could not locate figure_style.py")
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.collections import LineCollection
from matplotlib.patches import Ellipse
from PIL import Image
import ot
from scipy.linalg import expm, solve
from scipy.spatial.distance import cdist
from scipy.ndimage import gaussian_filter
from figure_style import (
    RED, BLUE, VIOLET, ORANGE, GRAY, LIGHT_GRAY, DIRAC_MARKER_SIZE,
    setup_matplotlib, figure_dir, save_pdf, remove_axes, box_axes,
    interp_color, draw_point_clouds, draw_transport_segments, padded_limits,
)
setup_matplotlib()
rng = np.random.default_rng(2027)

In [2]:
NAME='generative-diffusion-2d-forward-backward'; out=figure_dir(NAME)
weights=np.array([0.38,0.34,0.28]); means=np.array([[-1.05,-0.20],[0.82,-0.68],[0.42,0.98]]); scales=np.array([[0.18,0.26],[0.25,0.18],[0.20,0.22]])
def sample_data(n):
    comp=rng.choice(3,size=n,p=weights); return means[comp]+rng.normal(size=(n,2))*scales[comp]
def mix_density_score(z,t):
    z=np.atleast_2d(z); comps=[]; grads=[]
    for w,m,s in zip(weights,means,scales):
        mt=(1-t)*m; var=(1-t)**2*s**2 + t**2
        diff=z-mt; dens=w*np.exp(-0.5*((diff**2)/var).sum(axis=1))/(2*np.pi*np.sqrt(var.prod()))
        comps.append(dens); grads.append(-(diff/var)*dens[:,None])
    dens=np.sum(comps,axis=0)+1e-300; score=np.sum(grads,axis=0)/dens[:,None]
    return dens,score
def v_forward(z,t):
    _,score=mix_density_score(z,t); return -z/(1-t) - (t/(1-t))*score
xlim=(-1.72,1.42); ylim=(-1.30,1.48); gx=np.linspace(*xlim,120); gy=np.linspace(*ylim,110); X,Y=np.meshgrid(gx,gy); G=np.c_[X.ravel(),Y.ravel()]
fig,axes=plt.subplots(1,5,figsize=(5.65,1.26),gridspec_kw={'wspace':0.02})
for ax,t in zip(axes,[0,.25,.5,.75,.95]):
    den,_=mix_density_score(G,t); den=den.reshape(X.shape); den/=den.max()
    color=interp_color(float(t)); ax.contourf(X,Y,den,levels=np.linspace(.08,1,9),colors=[color],alpha=0.12)
    ax.contour(X,Y,den,levels=[.18,.34,.52,.70],colors=[color],linewidths=0.55,alpha=0.75)
    ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.set_aspect('equal'); remove_axes(ax)
save_pdf(fig,out/'forward-density.pdf',pad_inches=0.04); plt.close(fig)
# Backward probability-flow trajectories.
N=44; z=rng.normal(size=(N,2))*0.92; labels=np.argmin(cdist(z,means),axis=1); tgrid=np.linspace(.95,.05,95); paths=[z.copy()]
for k in range(len(tgrid)-1):
    t=tgrid[k]; dt=tgrid[k+1]-tgrid[k]; z=z+dt*v_forward(z,t); paths.append(z.copy())
paths=np.stack(paths,axis=1); colors=[tuple(c) for c in np.array([interp_color(i/2) for i in labels])]
fig,ax=plt.subplots(figsize=(2.75,2.20)); segs=[]; cols=[]
for i,path in enumerate(paths):
    for k in range(len(tgrid)-1):
        segs.append([path[k],path[k+1]]); cols.append((*colors[i],0.35))
ax.add_collection(LineCollection(segs,colors=cols,linewidths=0.52,zorder=1))
ax.scatter(paths[:,0,0],paths[:,0,1],s=DIRAC_MARKER_SIZE*.72,c=colors,marker='o',edgecolor='none',linewidth=0,alpha=0.78,zorder=3)
ax.scatter(paths[:,-1,0],paths[:,-1,1],s=DIRAC_MARKER_SIZE*.90,c=colors,marker='o',edgecolor='none',linewidth=0,alpha=0.92,zorder=4)
for m in means: ax.scatter([m[0]],[m[1]],s=DIRAC_MARKER_SIZE*1.3,color=RED,marker='o',edgecolor='none',linewidth=0,alpha=0.32,zorder=0)
ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.set_aspect('equal'); remove_axes(ax)
save_pdf(fig,out/'backward-trajectories.pdf',pad_inches=0.045); plt.close(fig)